# Assignment 4 — Sentiment Around AI Risk and the AI Equity ComplexAn event study around the **8 September 2026** post by Evan Hubinger (Anthropic)placing >10% probability on AI causing human extinction within a decade.This notebook calls the **same functions** the PDF report uses, so a number hereand a number in the report cannot disagree. Run the three scripts in `scripts/`first; everything below reads from cache.

In [ ]:
import sysfrom pathlib import Pathsys.path.insert(0, str(Path.cwd().parent))import pandas as pdfrom assignment4.src import config, exhibitsfrom assignment4.src.analysis import run_allpd.set_option("display.width", 160)pd.set_option("display.max_columns", 40)print(f"Event date : {config.EVENT_DATE}")print(f"Window     : {config.SAMPLE_START} to {config.SAMPLE_END}")print(f"Basket     : {', '.join(config.AI_BASKET)}")print(f"  semis        {', '.join(config.SEMICONDUCTOR)}")print(f"  hyperscalers {', '.join(config.HYPERSCALER)}")

## 1. Run the study`run_all` loads the cached raw data, scores both channels, aligns sentiment totrading days, runs the Granger tests and the media-bias comparisons, and writesevery exhibit to `outputs/`.

In [ ]:
results = run_all()tweets, news = results["tweets"], results["news"]print(f"tweets scored : {0 if tweets is None else len(tweets):,}")print(f"news scored   : {len(news):,}")print(f"trading days  : {results['window_panel']['date'].nunique()}")

## 2. The quality filtersThe brief mandates three filters on the social channel: no retweets, at least10 followers, at least 100 views. Worth confirming they actually bind ratherthan trusting that they ran.

In [ ]:
if tweets is not None:    print(f"min followers in retained set : {tweets['author_followers'].min():,}")    print(f"min views in retained set     : {tweets['view_count'].min():,}")    print(f"any retweets retained         : {bool(tweets.get('is_retweet', pd.Series([False])).any())}")    print()    print(tweets.groupby("family").agg(        n=("id", "size"),        mean_sentiment=("sentiment", "mean"),        median_followers=("author_followers", "median"),    ).round(3).to_string())else:    print("Social channel not harvested for this run.")

## 3. Sentiment trendThe brief asks for a clear trend over the window. Both channels are keptseparate: news aims at objectivity, social media is closer to raw sentiment.

In [ ]:
aligned = results["sentiment"]["aligned"]cols = [c for c in ["date", "x_ratio", "x_n", "news_ratio", "news_neg_share", "news_n"] if c in aligned]aligned[cols].round(3)

In [ ]:
figures = exhibits.build_all(results, config.OUTPUT_DIR)from IPython.display import Image, displayfor name in ("trend", "divergence"):    if name in figures:        display(Image(str(figures[name])))

### The two query families`ai_risk` measures fear about AI itself; `ai_company` measures sentiment towardthe AI trade. They are deliberately **not** blended — they can move in oppositedirections, and a combined index would net that out.

In [ ]:
sentiment = results["sentiment"]for family in ("ai_risk", "ai_company"):    key = f"x_daily_{family}"    if key in sentiment:        s = sentiment[key]        print(f"{family:12s} mean ratio {s['x_ratio'].mean():+.3f}  "              f"first {s['x_ratio'].iloc[0]:+.3f}  last {s['x_ratio'].iloc[-1]:+.3f}")

## 4. Granger causalityThe specification:$$A_{i,t} = \\alpha_i + \\sum_{k=1}^{p}\\beta_k A_{i,t-k} + \\sum_{k=1}^{p}\\gamma_k S_{t-k} + \\epsilon_{i,t}$$testing $H_0: \\gamma_1 = \\dots = \\gamma_p = 0$.**The panel specification is primary.** With 17 trading days a per-name VAR hasalmost no power. Pooling the eight names gives ~130 firm-days, and errors areclustered **by date** because the sentiment series is common across firms.

In [ ]:
print("Stationarity (ADF) — reported, not used to auto-difference:")display(results["granger"]["diagnostics"].round(4))

In [ ]:
print("PANEL (primary):")display(results["granger"]["panel"].round(4))print("\nSINGLE-SERIES (robustness only — underpowered at this sample length):")single = results["granger"]["single"]display(single.round(4) if len(single) else "none estimable")

## 5. Media bias — the COVID-19 parallelSacerdote, Sehgal & Cook (2020) found 91% of US major-outlet COVID stories werenegative vs 54% non-US and 65% scientific, and that negativity was *uncorrelated*with actual case trends. Three tests, run here on AI-risk coverage.

In [ ]:
bias = results["bias"]print("A. Negativity by outlet type")display(bias["by_bucket"].round(4))print("B. Coverage on up vs down days (the 5.5x asymmetry test)")display(bias["asymmetry"].round(4))print("C. Negativity vs fundamentals (is tone data-driven?)")display(bias["fundamentals"].round(4))

In [ ]:
if "by_region" in bias:    print("US vs non-US on the social channel — same platform, same event, different authors")    display(bias["by_region"].round(4))

In [ ]:
print("Most negative outlets (>=10 articles):")display(bias["by_outlet"].head(12).round(3))

In [ ]:
display(Image(str(figures["bias"])))display(Image(str(figures["returns"])))

## 6. The three required tables

In [ ]:
for exhibit in results["exhibits"]:    print(exhibit.to_markdown())    print("\n" + "-" * 90 + "\n")

## 7. Build the reportWrites `outputs/Assignment4_Report.pdf`.

In [ ]:
!python ../assignment4/build_report.py